In [ ]:
!pip install catboost
!pip install lime

In [ ]:
# =============================================================================
# TASK 2: DELIVERY SEVERITY PREDICTION
# E-Commerce Logistics - Explainable ML Framework
# Author  : Rakesh S Gautham (Student ID: 1196243)
# Thesis  : LJMU MSc AI/ML, March 2026
# =============================================================================
# 1. IMPORT LIBRARIES
# 2. LOAD DATASET
# 3. BUILD TARGET VARIABLE
# 4. EXPLORE THE TARGET VARIABLE
# 5. DATA PREPROCESSING
# 6. FEATURE ENGINEERING
# 7. FEATURE SELECTION
# 8. TRAIN-TEST SPLIT
# 9. MODEL SELECTION
# 10. CROSS-VALIDATION + TRAINING
# 11. FINAL TABLE
# 12. TEST DATA VALIDATION
# 13. SHAP (TREE MODELS ONLY)
# 14. LIME EXPLAINABILITY (TASK 2 — REGRESSION)
# 15. DONE
# =============================================================================

# -----------------------------------------------------------------------------
# STEP 1. IMPORT LIBRARIES
# -----------------------------------------------------------------------------
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, KFold, RandomizedSearchCV, cross_validate
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import xgboost as xgb
import lightgbm as lgb
import catboost as cb
from sklearn.svm import SVR
import shap
import lime
import lime.lime_tabular
import os
from time import time




from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.impute import SimpleImputer


os.makedirs("outputs_task2", exist_ok=True)

print("All libraries imported successfully.")


# -----------------------------------------------------------------------------
# STEP 2: LOAD DATASET
# -----------------------------------------------------------------------------
# Download from Kaggle and place the CSV in the same folder as this script
# https://www.kaggle.com/datasets/shashwatwork/dataco-smart-supply-chain-for-big-data-analysis

from google.colab import drive
drive.mount('/content/drive', force_remount=True)
scdata_df_raw=pd.read_csv('/content/drive/MyDrive/DataCoSupplyChainDataset.csv',encoding="latin1")
scdata_df = scdata_df_raw.copy()

# ── 2. EXPLORATORY DATA ANALYSIS (BRIEF) ─────────────────────────────────────
print("\n── 2. EDA Summary ──────────────────────────────────────────────────────")
print(f"   Shape of Dataset  : {scdata_df.shape,}")
print(f"   Missing values  : {scdata_df.isnull().sum().sum():,}")
print(f"   Duplicate rows  : {scdata_df.duplicated().sum():,}")
print("\nColumn names in dataset:")
print(scdata_df.columns.tolist())

print(f" Dropping ID & PI columns which are irrelevant, including the ones with missing values as they are not important")
drop_col_list = [
'Category Id',
'Customer Email',
'Customer Fname',
'Customer Id',
'Customer Lname',
'Customer Password',
'Department Id',
'Order Customer Id',
'Order Id',
'Order Item Cardprod Id',
'Order Item Id',
'Product Card Id',
'Product Category Id',
'Product Description',
'Product Image',
'Order Zipcode',
'Customer Zipcode',
'Product Status',
'Customer Street'
]

scdata_df = scdata_df.drop(columns=drop_col_list)

#Dropping rows that are having delivery status as Shipping Cancelled
scdata_df = scdata_df[scdata_df["Delivery Status"] != "Shipping canceled"]

# -----------------------------------------------------------------------------
# STEP 3: BUILD TARGET VARIABLE
# -----------------------------------------------------------------------------
# delay_days = Days for shipping (real)  -  Days for shipment (scheduled)
#
# "Days for shipping (real)"       → actual days taken to ship
# "Days for shipment (scheduled)"  → planned days for shipment
#
# Positive value = delivery took longer than planned  = LATE
# Negative value = delivery was faster than planned   = EARLY
# Zero            = delivered exactly on schedule     = ON TIME

scdata_df["delay_days"] = scdata_df["Days for shipping (real)"] - scdata_df["Days for shipment (scheduled)"]

# Remove rows where either column is missing
scdata_df = scdata_df.dropna(subset=["delay_days"])

print("\nTarget Variable — delay_days:")
print(scdata_df["delay_days"].describe())
print("\nSample counts:")
print("  Late deliveries  (delay > 0):", (scdata_df["delay_days"] > 0).sum())
print("  On time          (delay = 0):", (scdata_df["delay_days"] == 0).sum())
print("  Early deliveries (delay < 0):", (scdata_df["delay_days"] < 0).sum())

# -----------------------------------------------------------------------------
# STEP 4: EXPLORE THE TARGET VARIABLE
# -----------------------------------------------------------------------------

plt.figure(figsize=(13, 4))

plt.subplot(1, 2, 1)
plt.hist(scdata_df["delay_days"], bins=30, color="steelblue", edgecolor="white")
plt.axvline(scdata_df["delay_days"].mean(),   color="red",    linestyle="--", linewidth=2, label=f"Mean = {scdata_df['delay_days'].mean():.2f}")
plt.axvline(scdata_df["delay_days"].median(), color="orange", linestyle="--", linewidth=2, label=f"Median = {scdata_df['delay_days'].median():.2f}")
plt.xlabel("Delay Days")
plt.ylabel("Number of Orders")
plt.title("Distribution of Delay Days")
plt.legend()

plt.subplot(1, 2, 2)
scdata_df["delay_days"].value_counts().sort_index().plot(kind="bar", color="steelblue", edgecolor="white")
plt.xlabel("Delay Days")
plt.ylabel("Number of Orders")
plt.title("Frequency of Each Delay Value")

plt.tight_layout()
plt.savefig("plot_01_target_distribution.png", dpi=150)
plt.show()
print("Saved: plot_01_target_distribution.png")

# -----------------------------------------------------------------------------
# STEP 5: DATA PREPROCESSING
# -----------------------------------------------------------------------------
#Highly frequent values are retained and rest are grouped under 'Others'
def top_n_categories(df, col, n=15):
    top_categories = df[col].value_counts().nlargest(n).index
    return df[col].apply(lambda x: x if x in top_categories else 'Others')

# Order City
scdata_df['Order City Grouped'] = top_n_categories(scdata_df, 'Order City', 15)

# Category Name
scdata_df['Category Name Grouped'] = top_n_categories(scdata_df, 'Category Name', 15)

# Order Region
scdata_df['Order Region Grouped'] = top_n_categories(scdata_df, 'Order Region', 15)

# Order Country
scdata_df['Order Country Grouped'] = top_n_categories(scdata_df, 'Order Country', 10)

# Product Name
scdata_df['Product Name Grouped'] = top_n_categories(scdata_df, 'Product Name', 10)

# Customer City
scdata_df['Customer City Grouped'] = top_n_categories(scdata_df, 'Customer City', 10)

# Product Name
scdata_df['Customer State Grouped'] = top_n_categories(scdata_df, 'Customer State', 15)

# Product Name
scdata_df['Order State Grouped'] = top_n_categories(scdata_df, 'Order State', 15)

#Drop original cols
drop_original_cols = [
    'Category Name',
    'Product Name',
    'Order City',
    'Order Country',
    'Order Region',
    'Customer State',
    'Customer City',
    'Order State'
]

scdata_df = scdata_df.drop(columns=drop_original_cols)

print("Original categorical columns dropped. New DataFrame shape:", scdata_df.shape)
print("New columns added:", [col for col in scdata_df.columns if 'Grouped' in col])

#Post the correlation analysis the highly collinear columns are dropped
scdata_df.corr(numeric_only=True)
scdata_df = scdata_df.drop(columns=['Sales per customer', 'Sales', 'Product Price','Order Item Profit Ratio','Benefit per order'])
print("Highly collinear columns dropped . New DataFrame shape:", scdata_df.shape)

print("\n── OUTLIER HANDLING (IQR Capping / Winsorization) ─────────────────────")

# Columns to apply IQR capping
# Exclude binary/encoded columns and target
cap_cols = [
    "Order Item Discount",
    "Order Item Product Price",
    "Order Item Total",
    "Order Profit Per Order"
]

cap_cols = [c for c in cap_cols if c in scdata_df.columns]

# Store capping bounds for documentation
capping_log = []

for col in cap_cols:
    Q1  = scdata_df[col].quantile(0.25)
    Q3  = scdata_df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    # Count before capping
    n_lower = (scdata_df[col] < lower).sum()
    n_upper = (scdata_df[col] > upper).sum()

    # Apply capping
    scdata_df[col] = scdata_df[col].clip(lower=lower, upper=upper)

    capping_log.append({
        "Feature"      : col,
        "Lower Cap"    : round(lower, 2),
        "Upper Cap"    : round(upper, 2),
        "Capped Lower" : n_lower,
        "Capped Upper" : n_upper,
        "Total Capped" : n_lower + n_upper,
    })
    print(f"   {col:<40} lower={lower:.2f}  upper={upper:.2f}  "
          f"capped={n_lower + n_upper:,}")

capping_df = pd.DataFrame(capping_log)
capping_df.to_csv("outputs_task2/task2_outlier_capping_log.csv", index=False)
print("\n[✓] Capping log saved → outputs_task2/task2_outlier_capping_log.csv")

# -----------------------------------------------------------------------------
# STEP 6: FEATURE ENGINEERING
# -----------------------------------------------------------------------------
print("\n── 4. Feature Engineering ──────────────────────────────────────────────")

# --- 4a. Parse date columns ---
date_cols = [
    "order date (DateOrders)",
    "shipping date (DateOrders)"
]
for col in date_cols:
    if col in scdata_df.columns:
        scdata_df[col] = pd.to_datetime(scdata_df[col], errors="coerce")


# --- 4f. Temporal features ---
if "order date (DateOrders)" in scdata_df.columns:
    scdata_df["order_month"]     = scdata_df["order date (DateOrders)"].dt.month
    scdata_df["order_dayofweek"] = scdata_df["order date (DateOrders)"].dt.dayofweek



# -----------------------------------------------------------------------------
# STEP 7: FEATURE SELECTION
# -----------------------------------------------------------------------------
# Drop columns that are leakage risks or have no predictive value
LEAKAGE_COLS = [
    "Delivery Status", "Late_delivery_risk",  # direct leakage
    "order date (DateOrders)", "shipping date (DateOrders)","Days for shipping (real)","Days for shipment (scheduled)"
]
scdata_df = scdata_df.drop(columns=LEAKAGE_COLS)
print(f"   Feature count after engineering: {scdata_df.shape[1]}")

# -----------------------------------------------------------------------------
# STEP 8: TRAIN-TEST SPLIT
# -----------------------------------------------------------------------------

X = scdata_df.drop(columns=["delay_days"])
y = scdata_df["delay_days"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Detect feature types
num_feats = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_feats = X.select_dtypes(include=["object"]).columns.tolist()


# Tree-based models (no scaling)
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, num_feats),
    ("cat", categorical_transformer, cat_feats),
])

# SVR (with scaling)
numeric_transformer_scaled = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", RobustScaler())
])

preprocessor_scaled = ColumnTransformer([
    ("num", numeric_transformer_scaled, num_feats),
    ("cat", categorical_transformer, cat_feats),
])

# -----------------------------------------------------------------------------
# STEP 9: MODEL SELECTION
# -----------------------------------------------------------------------------

models = {
    "Linear Regression": Pipeline([
        ("prep", preprocessor),
        ("model", LinearRegression())
    ]),
    "Decision Tree": Pipeline([
        ("prep", preprocessor),
        ("model", DecisionTreeRegressor(max_depth=8, min_samples_leaf=20, random_state=42))
    ]),
    "Random Forest": Pipeline([
        ("prep", preprocessor),
        ("model", RandomForestRegressor(n_estimators=100, max_depth=8, min_samples_leaf=20, n_jobs=-1, random_state=42))
    ]),
    "XGBoost": Pipeline([
        ("prep", preprocessor),
        ("model", xgb.XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                   subsample=0.8, colsample_bytree=0.8,
                                   random_state=42, verbosity=0))
    ]),
    "LightGBM": Pipeline([
        ("prep", preprocessor),
        ("model", lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                    num_leaves=63, subsample=0.8,
                                    random_state=42, verbose=-1))
    ]),
    "CatBoost": Pipeline([
        ("prep", preprocessor),
        ("model", cb.CatBoostRegressor(iterations=300, learning_rate=0.05,
                                       depth=6, random_seed=42, verbose=0))
    ])
}

# SVR separately
svr_pipeline = Pipeline([
    ("prep", preprocessor_scaled),
    ("model", SVR(kernel="rbf", C=1.0, epsilon=0.5, gamma="scale"))
])

# -----------------------------------------------------------------------------
# STEP 10: CROSS-VALIDATION + TRAINING
# -----------------------------------------------------------------------------

print("\n── Cross-Validation + Training ─────────────────────────────")

CV_FOLDS = 5
kf = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=42)

results = {}
cv_results = {}

# Prepare SVR sample
svr_sample_size = min(20000, len(X_train))
rng = np.random.default_rng(42)
svr_sample_idx = rng.choice(len(X_train), size=svr_sample_size, replace=False)

# -----------------------------
# LOOP FOR TREE MODELS
# -----------------------------
for model_name, pipeline in models.items():

    print(f"\n🔹 {model_name}")

    # ---- Cross-validation ----
    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=kf,
        scoring={
            "MAE": "neg_mean_absolute_error",
            "RMSE": "neg_root_mean_squared_error",
            "R2": "r2"
        },
        n_jobs=-1
    )

    cv_mae = -scores["test_MAE"].mean()
    cv_rmse = -scores["test_RMSE"].mean()
    cv_r2 = scores["test_R2"].mean()

    print(f"   CV → MAE={cv_mae:.4f}, RMSE={cv_rmse:.4f}, R2={cv_r2:.4f}")


    # ---- Train full model ----

    t0 = time()
    pipeline.fit(X_train, y_train)
    train_time = time() - t0
    y_pred = pipeline.predict(X_test)

    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2   = r2_score(y_test, y_pred)

    print(f"   TEST → MAE={mae:.4f}, RMSE={rmse:.4f}, R2={r2:.4f}")

    results[model_name] = {
        "MAE"           : round(mae,        4),
        "RMSE"          : round(rmse,       4),
        "R2"            : round(r2,         4),
        "CV MAE"        : round(cv_mae,     4),
        "CV RMSE"       : round(cv_rmse,    4),
        "CV R2"         : round(cv_r2,      4),
        "Train Time (s)": round(train_time, 1),
        "_pipeline"     : pipeline,
        "_y_pred"       : y_pred,
    }

# -----------------------------
# SVR (SEPARATE)
# -----------------------------
print("\n🔹 SVR (sampled)")

X_svr = X_train.iloc[svr_sample_idx]
y_svr = y_train.iloc[svr_sample_idx]

scores = cross_validate(
    svr_pipeline,
    X_svr,
    y_svr,
    cv=kf,
    scoring={
        "MAE": "neg_mean_absolute_error",
        "RMSE": "neg_root_mean_squared_error",
        "R2": "r2"
    },
    n_jobs=-1
)

cv_mae = -scores["test_MAE"].mean()
cv_rmse = -scores["test_RMSE"].mean()
cv_r2 = scores["test_R2"].mean()

print(f"   CV → MAE={cv_mae:.4f}, RMSE={cv_rmse:.4f}, R2={cv_r2:.4f}")


t0 = time()
svr_pipeline.fit(X_svr, y_svr)
train_time = time() - t0
y_pred = svr_pipeline.predict(X_test)

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print(f"   TEST → MAE={mae:.4f}, RMSE={rmse:.4f}, R2={r2:.4f}")

results["SVR"] = {
    "MAE"           : round(mae,        4),
    "RMSE"          : round(rmse,       4),
    "R2"            : round(r2,         4),
    "CV MAE"        : round(cv_mae,     4),
    "CV RMSE"       : round(cv_rmse,    4),
    "CV R2"         : round(cv_r2,      4),
    "Train Time (s)": round(train_time, 1),
    "_pipeline"     : svr_pipeline,
    "_y_pred"       : y_pred,
}

# -----------------------------------------------------------------------------
# STEP 11: FINAL TABLE
# -----------------------------------------------------------------------------

metrics_cols = ["MAE", "RMSE", "R2", "CV MAE", "CV RMSE", "CV R2", "Train Time (s)"]

comparison_df = pd.DataFrame(
    {m: {k: results[m][k] for k in metrics_cols} for m in results}
).T.reset_index().rename(columns={"index": "Model"})

# Lower MAE = better for regression — ascending sort
comparison_df = comparison_df.sort_values("MAE", ascending=True).reset_index(drop=True)
comparison_df.insert(0, "Rank", comparison_df.index + 1)

print("\n" + "=" * 80)
print("  MODEL COMPARISON TABLE — TASK 2: DELAY SEVERITY PREDICTION")
print("=" * 80)
print(comparison_df.to_string(index=False))
print("=" * 80)

best_model_name = comparison_df.iloc[0]["Model"]
best_mae        = comparison_df.iloc[0]["MAE"]
best_r2         = comparison_df.iloc[0]["R2"]

print(f"\n  🏆 Best Model : {best_model_name}")
print(f"     MAE        : {best_mae:.4f}")
print(f"     R²         : {best_r2:.4f}")

comparison_df.to_csv("outputs_task2/task2_model_comparison.csv", index=False)
print("\n[✓] Saved → outputs_task2/task2_model_comparison.csv")


print("\n── Explainability (SHAP + LIME) ─────────────────────────────")

# -----------------------------------------------------------------------------
#  SELECT BEST MODEL
# -----------------------------------------------------------------------------

best_model_name = comparison_df.iloc[0]["Model"]
best_pipeline   = results[best_model_name]["_pipeline"]  # already fitted
best_y_pred     = results[best_model_name]["_y_pred"]    # already predicted

print(f"\n── SHAP Explainability — {best_model_name} ──────────────────────────────")
print(f"   Pulling fitted pipeline from results dict — no refit required.")

preprocessor  = best_pipeline.named_steps["prep"]
model         = best_pipeline.named_steps["model"]

X_train_transformed = preprocessor.transform(X_train)
X_test_transformed  = preprocessor.transform(X_test)
feature_names       = preprocessor.get_feature_names_out()

# #############################################################################
# STEP 12: — TEST DATA VALIDATION
# Delivery Delay Severity Prediction (Regression)
# #############################################################################

print("\n")
print("=" * 90)
print("  TEST DATA VALIDATION — TASK 2: DELIVERY DELAY SEVERITY PREDICTION")
print(f"  Best Model : {best_model_name}")
print("=" * 90)

best_pipeline_t2 = results[best_model_name]["_pipeline"]
best_y_pred_t2   = results[best_model_name]["_y_pred"]

# ── V1. test set metrics ───────────────────────────────────────────
print("\n   ── V1. Test Set Metric Confirmation ────────────────────────────────")

y_test_arr_t2 = np.array(y_test)

val_mae  = mean_absolute_error(y_test_arr_t2, best_y_pred_t2)
val_rmse = np.sqrt(mean_squared_error(y_test_arr_t2, best_y_pred_t2))
val_r2   = r2_score(y_test_arr_t2, best_y_pred_t2)

print(f"   Test MAE   : {val_mae:.4f}  (average error in days)")
print(f"   Test RMSE  : {val_rmse:.4f}  (penalises large errors)")
print(f"   Test R²    : {val_r2:.4f}  (variance explained by model)")

# ── V2. CV vs Test gap — overfitting check ────────────────────────────────────
print("\n   ── V2. Overfitting Check: CV vs Test Score Gap ─────────────────────")

cv_mae = results[best_model_name]["CV MAE"]
gap_t2 = abs(cv_mae - val_mae)

print(f"   CV MAE     : {cv_mae:.4f}")
print(f"   Test MAE   : {val_mae:.4f}")
print(f"   Gap        : {gap_t2:.4f}")

if gap_t2 < 0.05:
    verdict_t2 = "Excellent — model generalises well"
elif gap_t2 < 0.15:
    verdict_t2 = "Acceptable — minor gap, stable performance"
else:
    verdict_t2 = "Warning — notable gap, review regularisation"

print(f"   Verdict    : {verdict_t2}")

# ── V3. Residual analysis ─────────────────────────────────────────────────────
print("\n   ── V3. Residual Analysis ────────────────────────────────────────────")

residuals_t2 = y_test_arr_t2 - best_y_pred_t2

print(f"   Mean residual        : {residuals_t2.mean():.4f}  "
      f"(close to 0 = unbiased)")
print(f"   Std of residuals     : {residuals_t2.std():.4f}")
print(f"   Max overestimate     : {residuals_t2.min():.4f} days")
print(f"   Max underestimate    : {residuals_t2.max():.4f} days")
print(f"   % within ±1 day      : "
      f"{100*(np.abs(residuals_t2) <= 1).mean():.1f}%")
print(f"   % within ±2 days     : "
      f"{100*(np.abs(residuals_t2) <= 2).mean():.1f}%")

# ── V4. Actual vs predicted scatter ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter
axes[0].scatter(
    y_test_arr_t2, best_y_pred_t2,
    alpha=0.3, s=8, color="#1976D2"
)
mn = min(y_test_arr_t2.min(), best_y_pred_t2.min())
mx = max(y_test_arr_t2.max(), best_y_pred_t2.max())
axes[0].plot([mn, mx], [mn, mx], "r--", lw=1.5, label="Perfect prediction")
axes[0].set_xlabel("Actual Delay Days")
axes[0].set_ylabel("Predicted Delay Days")
axes[0].set_title(
    f"Actual vs Predicted — {best_model_name}\n"
    f"MAE={val_mae:.4f}  RMSE={val_rmse:.4f}  R²={val_r2:.4f}",
    fontsize=10, fontweight="bold"
)
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

# Residual histogram
axes[1].hist(residuals_t2, bins=30, color="#7B1FA2",
             edgecolor="white", alpha=0.85)
axes[1].axvline(0, color="red", linestyle="--", lw=1.5, label="Zero residual")
axes[1].axvline(residuals_t2.mean(), color="orange", linestyle="--",
                lw=1.5, label=f"Mean={residuals_t2.mean():.3f}")
axes[1].set_xlabel("Residual (Actual - Predicted)")
axes[1].set_ylabel("Frequency")
axes[1].set_title("Residual Distribution", fontsize=10, fontweight="bold")
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

fig.suptitle(
    f"Task 2 — Test Data Validation: {best_model_name}",
    fontsize=12, fontweight="bold"
)
plt.tight_layout()
plt.savefig(
    "outputs_task2/task2_validation_actual_vs_pred.png",
    dpi=150, bbox_inches="tight"
)
plt.close()
print("\n   [✓] Saved outputs_task2/task2_validation_actual_vs_pred.png")

# ── V5. Error severity breakdown ──────────────────────────────────────────────
print("\n   ── V5. Error Severity Breakdown ─────────────────────────────────────")

abs_res = np.abs(residuals_t2)
buckets = {
    "Exact (error = 0)"     : (abs_res == 0).sum(),
    "Within 1 day"          : (abs_res <= 1).sum(),
    "Within 2 days"         : (abs_res <= 2).sum(),
    "Error > 2 days"        : (abs_res > 2).sum(),
    "Error > 3 days"        : (abs_res > 3).sum(),
}
for label, count in buckets.items():
    pct = 100 * count / len(abs_res)
    print(f"   {label:<30} : {count:>7,}  ({pct:.1f}%)")

# ── V6. Validation summary ────────────────────────────────────────────────────
print(f"""
   ── V6. Validation Summary ───────────────────────────────────────────────
   Best Model       : {best_model_name}
   Test MAE         : {val_mae:.4f} days
   Test RMSE        : {val_rmse:.4f} days
   Test R²          : {val_r2:.4f}
   CV vs Test Gap   : {gap_t2:.4f}  ({verdict_t2.split(' — ')[0]})
   Mean Residual    : {residuals_t2.mean():.4f}  (bias check)
   Within ±1 day    : {100*(np.abs(residuals_t2) <= 1).mean():.1f}%

   Robustness Verdict : Model validated on held-out test set.
                        Residuals are approximately centred at zero.
                        Results are suitable for thesis reporting.
   ─────────────────────────────────────────────────────────────────────────
""")
print("=" * 90)

# -----------------------------------------------------------------------------
# STEP 13: SHAP (TREE MODELS ONLY)
# -----------------------------------------------------------------------------

print("\nGenerating SHAP explanations...")

# Sample data for speed
sample_size = min(5000, X_train_transformed.shape[0])
rng = np.random.default_rng(42)
sample_idx = rng.choice(X_train_transformed.shape[0], size=sample_size, replace=False)

X_sample = X_train_transformed[sample_idx]

# Use TreeExplainer for tree models
if best_model_name in ["Random Forest", "XGBoost", "LightGBM", "CatBoost", "Decision Tree"]:

    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_sample)

else:
    # Fallback (slow)
    explainer = shap.Explainer(model, X_sample)
    shap_values = explainer(X_sample)

# ---------------- SHAP SUMMARY ----------------
plt.figure()
shap.summary_plot(shap_values, X_sample, feature_names=feature_names, show=False)
plt.tight_layout()
plt.savefig("outputs_task2/shap_summary.png", dpi=150)
plt.close()

print("[✓] SHAP summary saved")

# ---------------- SHAP BAR ----------------
plt.figure()
shap.summary_plot(shap_values, X_sample, feature_names=feature_names, plot_type="bar", show=False)
plt.tight_layout()
plt.savefig("outputs_task2/shap_bar.png", dpi=150)
plt.close()

print("[✓] SHAP bar saved")

# ---------------- SHAP WATERFALL ----------------
try:
    idx = 0
    shap_value_single = shap_values[idx]

    plt.figure()
    shap.plots._waterfall.waterfall_legacy(
        explainer.expected_value,
        shap_value_single,
        feature_names=feature_names,
        max_display=15,
        show=False
    )
    plt.savefig("outputs_task2/shap_waterfall.png", dpi=150)
    plt.close()

    print("[✓] SHAP waterfall saved")

except Exception as e:
    print(f"[!] SHAP waterfall skipped: {e}")

# -----------------------------------------------------------------------------
# STEP 14: LIME EXPLAINABILITY (TASK 2 — REGRESSION)
# -----------------------------------------------------------------------------

# Operates in raw feature space with full pipeline wrapper
print("\n── LIME Explainability ─────────────────────────────────────────────────")


# ── Step 1: Build raw encoded arrays ────────────
all_features = num_feats + cat_feats

X_train_encoded = X_train[all_features].copy()
X_test_encoded  = X_test[all_features].copy()

# Identify categorical column indices for LIME
cat_indices = [all_features.index(c) for c in cat_feats if c in all_features]

# Build category mappings from training data
# Maps column index → list of unique string values seen in training
categorical_names = {}
for idx_c in cat_indices:
    col_name = all_features[idx_c]
    uniq = list(X_train_encoded[col_name].astype(str).unique())
    categorical_names[idx_c] = uniq

    # Encode as integer indices so LIME can perturb them numerically
    X_train_encoded[col_name] = X_train_encoded[col_name].astype(str).apply(
        lambda v: uniq.index(v) if v in uniq else 0
    )
    X_test_encoded[col_name] = X_test_encoded[col_name].astype(str).apply(
        lambda v: uniq.index(v) if v in uniq else 0
    )

X_train_np = X_train_encoded.values.astype(float)
X_test_np  = X_test_encoded.values.astype(float)

# ── Step 2: Pipeline wrapper for LIME ────────────────────────────────────
# LIME perturbs raw rows → wrapper decodes categories → full pipeline predicts
# Returns 1D array (regression mode — no class probabilities)
def lime_predict_fn_regression(raw_rows: np.ndarray) -> np.ndarray:
    df_temp = pd.DataFrame(raw_rows, columns=all_features)

    # Decode integer-encoded categoricals back to original string values
    for idx_c in cat_indices:
        col_name = all_features[idx_c]
        uniq     = categorical_names[idx_c]
        df_temp[col_name] = df_temp[col_name].apply(
            lambda v: uniq[int(round(v))] if 0 <= int(round(v)) < len(uniq)
                      else uniq[0]
        )

    # Full pipeline prediction (preprocessor + model)
    return best_pipeline.predict(df_temp)

# ── Step 3: Build LIME explainer on raw training data ────────────────────
lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data         = X_train_np,
    feature_names         = all_features,
    categorical_features  = cat_indices,
    categorical_names     = categorical_names,
    mode                  = "regression",   # ← key difference from Task 1/3
    random_state          = 42,
    discretize_continuous = True,
)

print(f"   LIME explainer built on {X_train_np.shape[0]:,} training samples.")

num_lime_features = min(15, len(all_features))
print(f"   LIME num_features set to : {num_lime_features}")

# ── Step 4: Explain three representative instances ────────────────────────
# Task 2 is regression so instead of TP/FP/FN we explain:
#   (a) A severely late delivery   — delay_days high positive
#   (b) An on-time/early delivery  — delay_days near zero or negative
#   (c) A moderately late delivery — delay_days in the middle

y_pred_best = best_pipeline.predict(X_test)
y_test_arr  = np.array(y_test)

instance_groups = {
    "severely_late"  : np.where(y_test_arr >= 4)[0],
    "on_time_early"  : np.where(y_test_arr <= 0)[0],
    "moderately_late": np.where((y_test_arr > 0) & (y_test_arr < 4))[0],
}

group_labels = {
    "severely_late"  : "Severely Late Delivery (delay ≥ 4 days)",
    "on_time_early"  : "On-Time / Early Delivery (delay ≤ 0 days)",
    "moderately_late": "Moderately Late Delivery (0 < delay < 4 days)",
}

for group_key, group_indices in instance_groups.items():

    if len(group_indices) == 0:
        print(f"   [!] No instances found for {group_key} — skipping")
        continue

    # Pick instance where model prediction is closest to actual value
    # i.e. the most representative, well-predicted instance in this group
    errors  = np.abs(y_pred_best[group_indices] - y_test_arr[group_indices])
    chosen  = group_indices[np.argmin(errors)]

    instance_raw   = X_test_np[chosen]
    actual_delay   = y_test_arr[chosen]
    pred_delay     = y_pred_best[chosen]

    print(f"\n   Explaining — {group_labels[group_key]}")
    print(f"   Actual delay : {actual_delay:.2f} days")
    print(f"   Predicted    : {pred_delay:.2f} days")

    exp = lime_explainer.explain_instance(
        data_row   = instance_raw,
        predict_fn = lime_predict_fn_regression,
        num_features = num_lime_features,
        num_samples  = 1000,
    )

    # Print top features
    print(f"   Top features:")
    for feat, weight in exp.as_list():
        print(f"   {feat:<45} {weight:>+.4f}")

    # Save feature weights to CSV
    lime_rows = []
    for feat, weight in exp.as_list():
        lime_rows.append({
            "Instance Group"   : group_key,
            "Actual Delay"     : round(actual_delay, 2),
            "Predicted Delay"  : round(pred_delay, 2),
            "Feature Condition": feat,
            "LIME Weight"      : round(weight, 4),
            "Direction"        : "increases delay" if weight > 0 else "decreases delay",
        })
    pd.DataFrame(lime_rows).to_csv(
        f"outputs_task2/task2_lime_{group_key}.csv",
        index=False,
    )
    print(f"   [✓] Saved outputs_task2/task2_lime_{group_key}.csv")

    # Save matplotlib figure
    fig = exp.as_pyplot_figure()
    fig.set_size_inches(12, 6)
    fig.suptitle(
        f"LIME Local Explanation — {group_labels[group_key]}\n"
        f"Model: {best_model_name} | "
        f"Actual: {actual_delay:.2f} days | "
        f"Predicted: {pred_delay:.2f} days",
        fontsize=10,
        fontweight="bold",
    )
    plt.tight_layout()
    plt.savefig(
        f"outputs_task2/task2_lime_{group_key}.png",
        dpi=150,
        bbox_inches="tight",
    )
    plt.close()
    print(f"   [✓] Saved outputs_task2/task2_lime_{group_key}.png")

# Save HTML for the moderately late instance (most common real-world case)
if len(instance_groups["moderately_late"]) > 0:
    errors  = np.abs(y_pred_best[instance_groups["moderately_late"]]
                     - y_test_arr[instance_groups["moderately_late"]])
    chosen  = instance_groups["moderately_late"][np.argmin(errors)]
    instance_raw = X_test_np[chosen]

    exp_html = lime_explainer.explain_instance(
        data_row     = instance_raw,
        predict_fn   = lime_predict_fn_regression,
        num_features = num_lime_features,
        num_samples  = 1000,
    )
    exp_html.save_to_file("outputs_task2/task2_lime_moderately_late.html")
    print("   [✓] Saved outputs_task2/task2_lime_moderately_late.html")

print("\n   ── LIME Summary ──────────────────────────────────────────────────────")
print("   Three instances explained: severely late / on-time / moderately late")
print("   Raw feature space used — consistent with Task 1 and Task 3.")
print("   Pipeline wrapper ensures preprocessing applied correctly per instance.")

# -----------------------------------------------------------------------------
# DONE
# -----------------------------------------------------------------------------
print("\n[✓] Explainability completed successfully.")